In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import libpysal
from libpysal.weights import Queen
import matplotlib.pyplot as plt
import geopandas as gpd
import scipy as sp
import arviz as az

RANDOM_SEED = 5781

# Transportation

## Data

In [2]:
tran_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_ONR_epa_climate.parquet'
tran_df = gpd.read_parquet(tran_par_file)
tran_wgt_file = '/work/hawkins_lab/vulcan/results/tran_weights.csv'
tran_wgt_df = pd.read_csv(tran_wgt_file)
# Create columns that assign decile numbers for each treatment variable
tran_df['treat_density'] = tran_df['d1a'] /1000
tran_df['treat_diversity'] = tran_df['d2b_e5mixa']
tran_df['treat_design'] = tran_df['d3a']
tran_df['treat_distance'] = tran_df['d4a'].replace(-99999,1500)  /1000
tran_df['treat_destination'] = tran_df['d5ar']  /1000

tran_df = pd.concat([tran_df,tran_wgt_df], axis=1)
tran_df = tran_df[(tran_df["value_weig"]>0) & (tran_df["totpop"]>0)]
tran_df["d1aa_cbsa"] = tran_df["totpop_cbsa"]/tran_df["ac_unpr_cbsa"] # true cbsa density

In [3]:
tran_df.groupby('cbsa').size()

cbsa
1.11          7
1.13         19
1.19         19
1.23         15
1.25         24
           ... 
49660.00    520
49700.00    111
49740.00    140
49780.00     75
49820.00      9
Length: 2013, dtype: int64

In [4]:
for c in ["treat_density","treat_diversity","treat_design","treat_distance","treat_destination", 
          "totpop_cbsa", "d1aa_cbsa","d2b_e5mix_cbsa","d3a_cbsa", "d4a_cbsa", "d5ar_cbsa"]:
    mu, sd = tran_df[c].mean(), tran_df[c].std()
    tran_df[c+"_z"] = (tran_df[c] - mu) / (sd if sd!=0 else 1.0)

for g in ["dens","div","des","dist","dest"]:
    mu, sd = tran_df[g].mean(), tran_df[g].std()
    tran_df["gps_"+g+"_z"] = (tran_df[g] - mu) / (sd if sd!=0 else 1.0)

tran_df["treat_density_ps"]   = tran_df["treat_density_z"]   * tran_df["gps_dens_z"]
tran_df["treat_diversity_ps"]  = tran_df["treat_diversity_z"] * tran_df["gps_div_z"]
tran_df["treat_design_ps"]     = tran_df["treat_design_z"]    * tran_df["gps_des_z"]
tran_df["treat_distance_ps"]  = tran_df["treat_distance_z"]  * tran_df["gps_dist_z"]
tran_df["treat_destination_ps"]= tran_df["treat_destination_z"] * tran_df["gps_dest_z"]

predictors = ["totpop_cbsa_z",
              "d1aa_cbsa_z",
              # "d1a_cbsa",
              "d2b_e5mix_cbsa_z",
              "d3a_cbsa_z",
              "d4a_cbsa_z", 
              "d5ar_cbsa_z",
              # "vmt_per_wo", # Add this one as the last of 3 transport-specific control variables from EPA data # Statistically insign
              "gps_dens_z",
              "gps_div_z",
              "gps_des_z",
              "gps_dist_z",
              "gps_dest_z",
              "treat_density_z",
              "treat_diversity_z",
              "treat_design_z",
              "treat_distance_z",
              "treat_destination_z",
              "treat_density_ps",
              "treat_diversity_ps",
              "treat_design_ps",
              "treat_distance_ps",
              "treat_destination_ps"]

X = tran_df.copy()

y = np.log(tran_df["value_weig"].replace(0, 10**-6) / tran_df["totpop"].replace(0, np.nan))

# X.loc[:,log_predictors] = np.log(X.loc[:,log_predictors].replace(0, 10**-6) ) # replace 0 with 10**-6

X1 = X.loc[:, predictors]
clusters = X['statefp']

## Model

In [5]:
X_fe = pd.get_dummies(tran_df["cbsa_eig"], prefix="metro", drop_first=True).astype(int)
# Combine with covariates
X1_fe = pd.concat([X1, X_fe], axis=1)

X1 = sm.add_constant(X1_fe)
weights = tran_df['totpop'] / tran_df['totpop'].mean()

modelWLS = sm.WLS(y, X1, weights=weights).fit(cov_type='cluster', cov_kwds={'groups': clusters})

In [6]:
print(modelWLS.summary())

                            WLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.275
Model:                            WLS   Adj. R-squared:                  0.274
Method:                 Least Squares   F-statistic:                 9.217e+09
Date:                Wed, 18 Feb 2026   Prob (F-statistic):          1.14e-226
Time:                        10:06:35   Log-Likelihood:            -3.0921e+05
No. Observations:              215309   AIC:                         6.191e+05
Df Residuals:                  214953   BIC:                         6.228e+05
Df Model:                         355                                         
Covariance Type:              cluster                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    1.3120 

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 355, but rank is 48
  warnings.warn('covariance of constraints does not have full '


In [7]:
modelOLS = sm.OLS(y, X1).fit(cov_type='cluster', cov_kwds={'groups': clusters})
print(modelOLS.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.269
Model:                            OLS   Adj. R-squared:                  0.268
Method:                 Least Squares   F-statistic:                -6.333e+08
Date:                Wed, 18 Feb 2026   Prob (F-statistic):               1.00
Time:                        10:06:48   Log-Likelihood:            -3.0206e+05
No. Observations:              215309   AIC:                         6.048e+05
Df Residuals:                  214953   BIC:                         6.085e+05
Df Model:                         355                                         
Covariance Type:              cluster                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    1.4073 

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 355, but rank is 48
  warnings.warn('covariance of constraints does not have full '


In [8]:
reg_table = modelOLS.summary2().tables[1]  # Coefficients table

# Export to LaTeX
limited = reg_table[['Coef.', 'Std.Err.','P>|z|']]
latex = limited.to_latex(float_format="%.3f")
print(latex)

\begin{tabular}{lrrr}
\toprule
 & Coef. & Std.Err. & P>|z| \\
\midrule
const & 1.407 & 0.057 & 0.000 \\
totpop_cbsa_z & -0.054 & 0.052 & 0.296 \\
d1aa_cbsa_z & 0.017 & 0.350 & 0.962 \\
d2b_e5mix_cbsa_z & 0.010 & 0.008 & 0.251 \\
d3a_cbsa_z & 0.009 & 0.013 & 0.503 \\
d4a_cbsa_z & -0.001 & 0.007 & 0.835 \\
d5ar_cbsa_z & 0.047 & 0.047 & 0.316 \\
gps_dens_z & -0.102 & 0.044 & 0.019 \\
gps_div_z & 0.021 & 0.010 & 0.043 \\
gps_des_z & -0.003 & 0.011 & 0.781 \\
gps_dist_z & 0.029 & 0.019 & 0.136 \\
gps_dest_z & 0.022 & 0.017 & 0.194 \\
treat_density_z & -0.187 & 0.062 & 0.003 \\
treat_diversity_z & 0.123 & 0.007 & 0.000 \\
treat_design_z & -0.449 & 0.032 & 0.000 \\
treat_distance_z & -0.001 & 0.027 & 0.980 \\
treat_destination_z & 0.324 & 0.115 & 0.005 \\
treat_density_ps & 0.029 & 0.020 & 0.148 \\
treat_diversity_ps & 0.039 & 0.005 & 0.000 \\
treat_design_ps & 0.007 & 0.007 & 0.318 \\
treat_distance_ps & 0.005 & 0.016 & 0.773 \\
treat_destination_ps & 0.015 & 0.018 & 0.404 \\
metro_4.0 & -0.

## Model diagnostics

In [9]:
w = Queen.from_dataframe(tran_df)
w.transform = 'r'

/tmp/ipykernel_2625728/2225362206.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(tran_df)


('WARNING: ', 18616, ' is an island (no neighbors)')
('WARNING: ', 29427, ' is an island (no neighbors)')
('WARNING: ', 32710, ' is an island (no neighbors)')
('WARNING: ', 84610, ' is an island (no neighbors)')
('WARNING: ', 89731, ' is an island (no neighbors)')
('WARNING: ', 94628, ' is an island (no neighbors)')
('WARNING: ', 126088, ' is an island (no neighbors)')
('WARNING: ', 134362, ' is an island (no neighbors)')
('WARNING: ', 137261, ' is an island (no neighbors)')
('WARNING: ', 150235, ' is an island (no neighbors)')
('WARNING: ', 172351, ' is an island (no neighbors)')
('WARNING: ', 207415, ' is an island (no neighbors)')


/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 27 disconnected components.
 There are 12 islands with ids: 18616, 29427, 32710, 84610, 89731, 94628, 126088, 134362, 137261, 150235, 172351, 207415.
  W.__init__(self, neighbors, ids=ids, **kw)


In [10]:
from esda.moran import Moran

residuals = modelWLS.resid
mi = Moran(residuals, w)
print(mi.I, mi.p_sim)

0.368657715811432 0.001


In [11]:
residuals = modelOLS.resid
mi = Moran(residuals, w)
print(mi.I, mi.p_sim)

0.37010592509229717 0.001


# Residential Energy

## Data

In [12]:
res_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_RES_epa_climate.parquet'
res_df = gpd.read_parquet(res_par_file)
res_wgt_file = '/work/hawkins_lab/vulcan/results/res_weights.csv'
res_wgt_df = pd.read_csv(res_wgt_file)
# Create columns that assign decile numbers for each treatment variable
res_df['treat_density'] = res_df['d1a']
res_df['treat_diversity'] = res_df['d2b_e5mixa']

res_df = pd.concat([res_df,res_wgt_df], axis=1)
res_df = res_df[(res_df["value_weig"]>0) & (res_df["totpop"]>0)]
res_df["d1aa_cbsa"] = res_df["totpop_cbsa"]/res_df["ac_unpr_cbsa"] # true cbsa density

for c in ["treat_density","treat_diversity", "totpop_cbsa", "d1aa_cbsa","d2b_e5mix_cbsa"]:
    mu, sd = res_df[c].mean(), res_df[c].std()
    res_df[c+"_z"] = (res_df[c] - mu) / (sd if sd!=0 else 1.0)

for g in ["dens","div"]:
    mu, sd = res_df[g].mean(), res_df[g].std()
    res_df["gps_"+g+"_z"] = (res_df[g] - mu) / (sd if sd!=0 else 1.0)

res_df["treat_density_ps"]   = res_df["treat_density_z"]   * res_df["gps_dens_z"]
res_df["treat_diversity_ps"]  = res_df["treat_diversity_z"] * res_df["gps_div_z"]

predictors = ["totpop_cbsa_z",
              "d1aa_cbsa_z",
              # "d1a_cbsa",
              "d2b_e5mix_cbsa_z",
              # "vmt_per_wo", # Add this one as the last of 3 transport-specific control variables from EPA data # Statistically insign
              "gps_dens_z",
              "gps_div_z",
              "treat_density_z",
              "treat_diversity_z",
              "treat_density_ps",
              "treat_diversity_ps"]

X = res_df.copy()

y = np.log(res_df["value_weig"].replace(0, 10**-6) / res_df["totpop"].replace(0, np.nan)) # make per capita

X1 = X.loc[:, predictors]
clusters = X['statefp']

## Model

In [13]:
X_fe = pd.get_dummies(res_df["cbsa_eig"], prefix="metro", drop_first=True).astype(int)
# Combine with covariates
X1_fe = pd.concat([X1, X_fe], axis=1)
X1 = sm.add_constant(X1_fe)
weights = res_df['totpop'] / res_df['totpop'].mean()

modelWLS = sm.WLS(y, X1, weights=weights).fit(cov_type='cluster', cov_kwds={'groups': clusters})

In [14]:
print(modelWLS.summary())

                            WLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.676
Model:                            WLS   Adj. R-squared:                  0.675
Method:                 Least Squares   F-statistic:                 1.987e+05
Date:                Wed, 18 Feb 2026   Prob (F-statistic):          3.63e-114
Time:                        10:11:43   Log-Likelihood:            -2.5458e+05
No. Observations:              215249   AIC:                         5.098e+05
Df Residuals:                  214905   BIC:                         5.134e+05
Df Model:                         343                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -1.3129      0

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 343, but rank is 41
  warnings.warn('covariance of constraints does not have full '


In [15]:
modelOLS = sm.OLS(y, X1).fit(cov_type='cluster', cov_kwds={'groups': clusters})
print(modelOLS.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.638
Model:                            OLS   Adj. R-squared:                  0.638
Method:                 Least Squares   F-statistic:                 7.474e+05
Date:                Wed, 18 Feb 2026   Prob (F-statistic):          5.67e-128
Time:                        10:12:25   Log-Likelihood:            -2.3881e+05
No. Observations:              215249   AIC:                         4.783e+05
Df Residuals:                  214905   BIC:                         4.818e+05
Df Model:                         343                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -1.0764      0

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 343, but rank is 41
  warnings.warn('covariance of constraints does not have full '


In [16]:
reg_table = modelOLS.summary2().tables[1]  # Coefficients table

# Export to LaTeX
limited = reg_table[['Coef.', 'Std.Err.','P>|z|']]
latex = limited.to_latex(float_format="%.3f")
print(latex)

\begin{tabular}{lrrr}
\toprule
 & Coef. & Std.Err. & P>|z| \\
\midrule
const & -1.076 & 0.017 & 0.000 \\
totpop_cbsa_z & 0.029 & 0.021 & 0.171 \\
d1aa_cbsa_z & -0.033 & 0.230 & 0.887 \\
d2b_e5mix_cbsa_z & -0.006 & 0.014 & 0.647 \\
gps_dens_z & 0.015 & 0.031 & 0.623 \\
gps_div_z & 0.014 & 0.008 & 0.065 \\
treat_density_z & -0.120 & 0.046 & 0.009 \\
treat_diversity_z & 0.008 & 0.006 & 0.180 \\
treat_density_ps & -0.029 & 0.020 & 0.150 \\
treat_diversity_ps & 0.010 & 0.005 & 0.051 \\
metro_4.0 & 0.074 & 0.007 & 0.000 \\
metro_5.0 & 0.650 & 0.009 & 0.000 \\
metro_6.0 & 0.721 & 0.024 & 0.000 \\
metro_8.0 & 1.579 & 0.012 & 0.000 \\
metro_10.0 & 1.201 & 0.009 & 0.000 \\
metro_12.0 & -1.782 & 0.011 & 0.000 \\
metro_13.0 & -0.206 & 0.011 & 0.000 \\
metro_16.0 & 1.050 & 0.054 & 0.000 \\
metro_17.0 & 1.832 & 0.010 & 0.000 \\
metro_18.0 & 1.335 & 0.012 & 0.000 \\
metro_19.0 & 1.690 & 0.010 & 0.000 \\
metro_20.0 & 1.503 & 0.006 & 0.000 \\
metro_21.0 & 0.426 & 0.014 & 0.000 \\
metro_22.0 & 0.179 & 0

## Model diagnostics

In [17]:
w = Queen.from_dataframe(res_df)
w.transform = 'r'

/tmp/ipykernel_2625728/682235214.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(res_df)


('WARNING: ', 18612, ' is an island (no neighbors)')
('WARNING: ', 32704, ' is an island (no neighbors)')
('WARNING: ', 84580, ' is an island (no neighbors)')
('WARNING: ', 89701, ' is an island (no neighbors)')
('WARNING: ', 94598, ' is an island (no neighbors)')
('WARNING: ', 96633, ' is an island (no neighbors)')
('WARNING: ', 134310, ' is an island (no neighbors)')
('WARNING: ', 137205, ' is an island (no neighbors)')
('WARNING: ', 150177, ' is an island (no neighbors)')
('WARNING: ', 172293, ' is an island (no neighbors)')
('WARNING: ', 207355, ' is an island (no neighbors)')


/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 26 disconnected components.
 There are 11 islands with ids: 18612, 32704, 84580, 89701, 94598, 96633, 134310, 137205, 150177, 172293, 207355.
  W.__init__(self, neighbors, ids=ids, **kw)


In [18]:
residuals = modelWLS.resid
mi = Moran(residuals, w)
print(mi.I, mi.p_sim)

0.26179564241274483 0.001


In [19]:
residuals = modelOLS.resid
mi = Moran(residuals, w)
print(mi.I, mi.p_sim)

0.2580122341119262 0.001


## Residential Electricity

## Data

In [20]:
elec_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_SC2_epa_climate.parquet'
elec_df = gpd.read_parquet(elec_par_file)
elec_wgt_file = '/work/hawkins_lab/vulcan/results/elec_weights.csv'
elec_wgt_df = pd.read_csv(elec_wgt_file)
# Create columns that assign decile numbers for each treatment variable
elec_df['treat_density'] = elec_df['d1a']
elec_df['treat_diversity'] = elec_df['d2b_e5mixa']

elec_df = pd.concat([elec_df,elec_wgt_df], axis=1)
elec_df = elec_df[(elec_df["res_tc"]>0) & (elec_df["totpop"]>0)]
elec_df["d1aa_cbsa"] = elec_df["totpop_cbsa"]/elec_df["ac_unpr_cbsa"] # true cbsa density

for c in ["treat_density","treat_diversity", "totpop_cbsa", "d1aa_cbsa","d2b_e5mix_cbsa"]:
    mu, sd = elec_df[c].mean(), elec_df[c].std()
    elec_df[c+"_z"] = (elec_df[c] - mu) / (sd if sd!=0 else 1.0)

for g in ["dens","div"]:
    mu, sd = elec_df[g].mean(), elec_df[g].std()
    elec_df["gps_"+g+"_z"] = (elec_df[g] - mu) / (sd if sd!=0 else 1.0)

elec_df["treat_density_ps"]   = elec_df["treat_density_z"]   * elec_df["gps_dens_z"]
elec_df["treat_diversity_ps"]  = elec_df["treat_diversity_z"] * elec_df["gps_div_z"]

predictors = ["totpop_cbsa_z",
              "d1aa_cbsa_z",
              # "d1a_cbsa",
              "d2b_e5mix_cbsa_z",
              # "vmt_per_wo", # Add this one as the last of 3 transport-specific control variables from EPA data # Statistically insign
              "gps_dens_z",
              "gps_div_z",
              "treat_density_z",
              "treat_diversity_z",
              "treat_density_ps",
              "treat_diversity_ps"]

X = elec_df.copy()

y = np.log(elec_df["res_tc"].replace(0, 10**-6) / elec_df["totpop"].replace(0, np.nan)) # make per capita

X1 = X.loc[:, predictors]
clusters = X['statefp']

## Model

In [21]:
X_fe = pd.get_dummies(elec_df["cbsa_eig"], prefix="metro", drop_first=True).astype(int)
# Combine with covariates
X1_fe = pd.concat([X1, X_fe], axis=1)
X1 = sm.add_constant(X1_fe)
weights = elec_df['totpop'] / elec_df['totpop'].mean()

modelWLS = sm.WLS(y, X1, weights=weights).fit(cov_type='cluster', cov_kwds={'groups': clusters})

In [22]:
print(modelWLS.summary())

                            WLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.679
Model:                            WLS   Adj. R-squared:                  0.678
Method:                 Least Squares   F-statistic:                -8.995e+11
Date:                Wed, 18 Feb 2026   Prob (F-statistic):               1.00
Time:                        10:17:08   Log-Likelihood:            -1.6263e+05
No. Observations:              216623   AIC:                         3.259e+05
Df Residuals:                  216274   BIC:                         3.295e+05
Df Model:                         348                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -0.3225      0

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 348, but rank is 41
  warnings.warn('covariance of constraints does not have full '


In [23]:
modelOLS = sm.OLS(y, X1).fit(cov_type='cluster', cov_kwds={'groups': clusters})
print(modelOLS.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.648
Model:                            OLS   Adj. R-squared:                  0.648
Method:                 Least Squares   F-statistic:                 9.904e+11
Date:                Wed, 18 Feb 2026   Prob (F-statistic):          3.85e-286
Time:                        10:17:13   Log-Likelihood:            -1.5830e+05
No. Observations:              216623   AIC:                         3.173e+05
Df Residuals:                  216274   BIC:                         3.209e+05
Df Model:                         348                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -0.2400      0

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 348, but rank is 41
  warnings.warn('covariance of constraints does not have full '


In [24]:
reg_table = modelOLS.summary2().tables[1]  # Coefficients table

# Export to LaTeX
limited = reg_table[['Coef.', 'Std.Err.','P>|z|']]
latex = limited.to_latex(float_format="%.3f")
print(latex)

\begin{tabular}{lrrr}
\toprule
 & Coef. & Std.Err. & P>|z| \\
\midrule
const & -0.240 & 0.014 & 0.000 \\
totpop_cbsa_z & 0.005 & 0.021 & 0.792 \\
d1aa_cbsa_z & -0.138 & 0.218 & 0.528 \\
d2b_e5mix_cbsa_z & -0.015 & 0.013 & 0.233 \\
gps_dens_z & 0.013 & 0.011 & 0.242 \\
gps_div_z & -0.024 & 0.005 & 0.000 \\
treat_density_z & -0.068 & 0.028 & 0.014 \\
treat_diversity_z & 0.037 & 0.005 & 0.000 \\
treat_density_ps & 0.001 & 0.007 & 0.872 \\
treat_diversity_ps & -0.035 & 0.004 & 0.000 \\
metro_2.0 & -0.163 & 0.012 & 0.000 \\
metro_4.0 & -0.737 & 0.005 & 0.000 \\
metro_5.0 & 0.169 & 0.006 & 0.000 \\
metro_6.0 & -1.195 & 0.005 & 0.000 \\
metro_8.0 & 0.050 & 0.010 & 0.000 \\
metro_10.0 & -0.321 & 0.007 & 0.000 \\
metro_12.0 & -0.008 & 0.006 & 0.155 \\
metro_13.0 & 0.182 & 0.010 & 0.000 \\
metro_15.0 & -0.054 & 0.007 & 0.000 \\
metro_16.0 & -0.696 & 0.011 & 0.000 \\
metro_17.0 & -0.430 & 0.009 & 0.000 \\
metro_18.0 & -0.355 & 0.007 & 0.000 \\
metro_19.0 & -0.117 & 0.010 & 0.000 \\
metro_20.0 & 0

## Model diagnostics

In [25]:
w = Queen.from_dataframe(elec_df)
w.transform = 'r'

/tmp/ipykernel_2625728/2594203160.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(elec_df)


('WARNING: ', 3883, ' is an island (no neighbors)')
('WARNING: ', 85955, ' is an island (no neighbors)')
('WARNING: ', 91075, ' is an island (no neighbors)')
('WARNING: ', 95971, ' is an island (no neighbors)')
('WARNING: ', 98005, ' is an island (no neighbors)')
('WARNING: ', 127428, ' is an island (no neighbors)')
('WARNING: ', 135701, ' is an island (no neighbors)')
('WARNING: ', 138599, ' is an island (no neighbors)')
('WARNING: ', 140953, ' is an island (no neighbors)')
('WARNING: ', 151571, ' is an island (no neighbors)')
('WARNING: ', 173687, ' is an island (no neighbors)')


/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 34 disconnected components.
 There are 11 islands with ids: 3883, 85955, 91075, 95971, 98005, 127428, 135701, 138599, 140953, 151571, 173687.
  W.__init__(self, neighbors, ids=ids, **kw)


In [26]:
residuals = modelWLS.resid
mi = Moran(residuals, w)
print(mi.I, mi.p_sim)

0.43354889885230036 0.001


In [27]:
residuals = modelOLS.resid
mi = Moran(residuals, w)
print(mi.I, mi.p_sim)

0.43141317078960684 0.001
